# Python named configuration exploration

Inspect the live Python profile contract and exercise its actual save, environment qualification, selection and execution paths. This notebook uses task-owned profiles and local fixture projects. Existing notebooks, environments and defaults are preserved.

The current contract supports **existing**, **requirements**, **pyproject** and **uv_project** adapters. Preview/Save do not install dependencies. Explicit preparation of an existing interpreter qualifies it without installation. Named Python currently supports plain execution; its Arrow/port API is disabled by contract.

## 1. Factory control and isolated fixture projects

Observe the existing factory interpreter and `ipykernel`, then create two local-only projects. Both profiles will use that already-prepared interpreter, while their project roots and Python processes should remain separate. No package or environment files are modified.

In [2]:
import hashlib, importlib.metadata, json, os, pathlib, sys, tempfile

python_named_lab = {'notebook_path': '/Users/kevintruong/project/spur-notebook/2026-11-18-python-named-configuration-exploration.ipynb'}
python_named_lab['factory'] = {'executable':sys.executable,'version':sys.version.split()[0],'prefix':sys.prefix,'cwd':os.getcwd(),'pid':os.getpid(),'ipykernel':importlib.metadata.version('ipykernel')}
base = pathlib.Path(tempfile.mkdtemp(prefix='spur-python-named-',dir='/private/tmp'))
projects = {}
for name in ('a','b'):
    root=base/name
    root.mkdir()
    (root/'module_from_project.py').write_text('PROJECT_MARKER = '+repr(name)+'\n',encoding='utf-8')
    (root/'requirements.txt').write_text('# Local-only diagnostic fixture; no dependencies.\n',encoding='utf-8')
    (root/'pyproject.toml').write_text('[project]\nname = "spur-python-probe-'+name+'"\nversion = "0.0.0"\nrequires-python = ">=3.12"\ndependencies = []\n',encoding='utf-8')
    projects[name]={'root':str(root),'config':{'project_root':os.path.relpath(root,pathlib.Path(python_named_lab['notebook_path']).parent),'environment':{'kind':'existing','interpreter':{'kind':'project_relative','path':os.path.relpath(sys.executable,root)}}},'hashes':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(root.iterdir())}}
python_named_lab['projects']=projects
print(json.dumps(python_named_lab,indent=2))

{
  "notebook_path": "/Users/kevintruong/project/spur-notebook/2026-11-18-python-named-configuration-exploration.ipynb",
  "factory": {
    "executable": "/Users/kevintruong/.spur/jupyter/venv/bin/python",
    "version": "3.12.10",
    "prefix": "/Users/kevintruong/.spur/jupyter/venv",
    "cwd": "/Users/kevintruong/project/spur-notebook",
    "pid": 91094,
    "ipykernel": "7.4.0"
  },
  "projects": {
    "a": {
      "root": "/private/tmp/spur-python-named-_47wrdz6/a",
      "config": {
        "project_root": "../../../../private/tmp/spur-python-named-_47wrdz6/a",
        "environment": {
          "kind": "existing",
          "interpreter": {
            "kind": "project_relative",
            "path": "../../../../Users/kevintruong/.spur/jupyter/venv/bin/python"
          }
        }
      },
      "hashes": {
        "module_from_project.py": "82f8ea6f8cf02bd219474924ebc6ffb2b72ecf924b021d6bd9edc02c206f48db",
        "pyproject.toml": "931423f9ea0fad5553ba92b699b17471b34fc91bd8c9

In [2]:
import importlib.metadata, json, os, pathlib, sys
import module_from_project

profile_state = {'owner':'a','value':41}
python_profile_a = {
    'profile':'py-probe-a','pid':os.getpid(),'cwd':os.getcwd(),
    'executable':sys.executable,'version':sys.version.split()[0],
    'prefix':sys.prefix,'project_marker':module_from_project.PROJECT_MARKER,
    'ipykernel':importlib.metadata.version('ipykernel'),
    'state':profile_state
}
assert module_from_project.PROJECT_MARKER == 'a'
print(json.dumps(python_profile_a,indent=2))

{
  "profile": "py-probe-a",
  "pid": 93662,
  "cwd": "/private/tmp/spur-python-named-_47wrdz6/a",
  "executable": "/Users/kevintruong/.spur/jupyter/venv/bin/python",
  "version": "3.12.10",
  "prefix": "/Users/kevintruong/.spur/jupyter/venv",
  "project_marker": "a",
  "ipykernel": "7.4.0",
  "state": {
    "owner": "a",
    "value": 41
  }
}


## 2. Two named profiles and process isolation

`py-probe-a` ran in project A using the explicitly qualified existing interpreter. A second task-owned profile, `py-probe-b`, points to project B using the same interpreter. This tests separate kernel processes and globals, not separate virtual environments.

The next cell selects B explicitly. It must import B's module and must not inherit the `profile_state` variable defined in A.

In [2]:
import importlib.metadata, json, os, pathlib, sys
sys.dont_write_bytecode = True
assert 'profile_state' not in globals(), 'State leaked from another profile'
import module_from_project
assert module_from_project.PROJECT_MARKER == 'b'
profile_state = {'owner':'b','value':99}
python_profile_b = {'profile':'py-probe-b','pid':os.getpid(),'cwd':os.getcwd(),'executable':sys.executable,'version':sys.version.split()[0],'project_marker':module_from_project.PROJECT_MARKER,'other_profile_state_absent':True,'state':profile_state}
print(json.dumps(python_profile_b,indent=2))

{
  "profile": "py-probe-b",
  "pid": 94915,
  "cwd": "/private/tmp/spur-python-named-_47wrdz6/b",
  "executable": "/Users/kevintruong/.spur/jupyter/venv/bin/python",
  "version": "3.12.10",
  "project_marker": "b",
  "other_profile_state_absent": true,
  "state": {
    "owner": "b",
    "value": 99
  }
}


## 3. Notebook default and explicit override

The notebook's Python default is temporarily set to `py-probe-a`. The next cell has no per-cell selector and should reuse A's process/state. The following cell explicitly selects B and should retain B's separate process/state. The default will be restored to its original factory setting after these probes.

In [3]:
import json, os
assert profile_state == {'owner':'a','value':41}, 'Default did not reuse profile A state'
assert module_from_project.PROJECT_MARKER == 'a'
print(json.dumps({'selection':'inherited notebook default','expected_profile':'py-probe-a','pid':os.getpid(),'cwd':os.getcwd(),'state':profile_state,'project_marker':module_from_project.PROJECT_MARKER},indent=2))

{
  "selection": "inherited notebook default",
  "expected_profile": "py-probe-a",
  "pid": 93662,
  "cwd": "/private/tmp/spur-python-named-_47wrdz6/a",
  "state": {
    "owner": "a",
    "value": 41
  },
  "project_marker": "a"
}


In [3]:
import json, os
assert profile_state == {'owner':'b','value':99}, 'Explicit selector did not reuse profile B state'
assert module_from_project.PROJECT_MARKER == 'b'
print(json.dumps({'selection':'explicit B while notebook default is A','expected_profile':'py-probe-b','pid':os.getpid(),'cwd':os.getcwd(),'state':profile_state,'project_marker':module_from_project.PROJECT_MARKER},indent=2))

{
  "selection": "explicit B while notebook default is A",
  "expected_profile": "py-probe-b",
  "pid": 94915,
  "cwd": "/private/tmp/spur-python-named-_47wrdz6/b",
  "state": {
    "owner": "b",
    "value": 99
  },
  "project_marker": "b"
}


## 4. Environment adapters and preservation

The `existing` adapter was explicitly qualified and used for both successful runs. The qualification plan declared no downloads or build hooks and only a private readiness receipt as its write scope.

The other adapters were inspected through passive Preview only. No environment was built or installed for requirements, pyproject or uv. In particular, a uv project must have the required lock/prepared prerequisites; this fixture intentionally has no `uv.lock`.

The original notebook default has been restored to factory Python. To replay the historical inherited-default cell above, temporarily select `py-probe-a` as the notebook's Python default. The explicitly bound A/B cells can be run independently once their recorded setup cells have initialized state.

The next factory cell checks that original fixture source files remain unchanged and that the named profiles' variables did not leak into the factory process. Generated Python bytecode, if present, is reported separately.

In [3]:
import hashlib, json, os, pathlib, sys
project_preservation={}
for name,project in python_named_lab['projects'].items():
    root=pathlib.Path(project['root'])
    after={f:hashlib.sha256((root/f).read_bytes()).hexdigest() for f in project['hashes']}
    generated=sorted(str(p.relative_to(root)) for p in root.rglob('*') if p.is_file() and str(p.relative_to(root)) not in project['hashes'])
    project_preservation[name]={'original_sources_unchanged':after==project['hashes'],'after':after,'generated_files':generated}
    assert after==project['hashes']
assert 'profile_state' not in globals(), 'Named state leaked into factory'
factory_final={'pid':os.getpid(),'executable':sys.executable,'cwd':os.getcwd(),'named_state_absent':True,'project_preservation':project_preservation}
print(json.dumps(factory_final,indent=2))

{
  "pid": 91094,
  "executable": "/Users/kevintruong/.spur/jupyter/venv/bin/python",
  "cwd": "/Users/kevintruong/project/spur-notebook",
  "named_state_absent": true,
  "project_preservation": {
    "a": {
      "original_sources_unchanged": true,
      "after": {
        "module_from_project.py": "82f8ea6f8cf02bd219474924ebc6ffb2b72ecf924b021d6bd9edc02c206f48db",
        "pyproject.toml": "931423f9ea0fad5553ba92b699b17471b34fc91bd8c90eec03140bbee719a54a",
        "requirements.txt": "9c15bb2dfeb295cf5383aafa11a5333f0d7c6bc2dfda76785bf5f926abb5a51c"
      },
      "generated_files": [
        "__pycache__/module_from_project.cpython-312.pyc"
      ]
    },
    "b": {
      "original_sources_unchanged": true,
      "after": {
        "module_from_project.py": "db726d822d657fc74a099381477ac265cf8ffda1311b0371daded33d3f6dba2f",
        "pyproject.toml": "f99a2df8ce7e798edd6f96d9328d669e71e83336e7eec058434ae4e7bdddbdf7",
        "requirements.txt": "9c15bb2dfeb295cf5383aafa11a5333f0d7c6b

# Findings and replay notes

**Named Python execution passed through MCP in this build.** Six code cells completed successfully across factory Python and two named profiles.

| Behavior | Observed result |
|---|---|
| Named profile schema | Live Python schema, shared registry version 4 |
| Preview | Passive; no profile UUID allocated |
| Save | Stable UUIDs, no installation or runtime startup |
| Explicit existing-interpreter preparation | Reviewed plan, no downloads/build hooks, qualification receipt saved |
| Profile A | Separate process, configured project A cwd and module |
| Profile B | Different process, configured project B cwd and module; A globals absent |
| Notebook default | Temporarily selected A; inherited cell reused A's existing state |
| Explicit selector | B overrode default A and retained B's state |
| Restoration | Notebook default returned to factory Python |
| Factory isolation | Named globals absent from factory process |
| Source preservation | All six original project files unchanged; one Python bytecode cache appeared in A |
| Arrow/data ports | Explicitly disabled for named Python; plain execution only |

The two named profiles use the **same existing Python 3.12.10 environment**, with ipykernel 7.4.0. This verifies project/process selection and state isolation; it does not verify separate dependency environments. Their current processes remain available for exploration.

Requirements and pyproject modes were previewed but not prepared or installed. The uv preview correctly reported its missing `uv.lock`; no lock was created. Local-binding mode, managed environment builds, dependency changes, restart recovery, human UI interactions and named Arrow publication were not tested.

For replay: run the factory fixture cell only if you intend to create new project roots, then update the saved profiles to those roots. The saved A/B profiles currently point to the original temporary projects printed above. Run A and B's explicitly selected setup cells before their state checks. The historical inherited-default check requires temporarily choosing A as notebook default; the delivered notebook restores factory default.

Evidence and the captured notebook are saved under `/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-18-python-named-configuration/`.